# Import libraries

In [1]:
import os
import cohere
from dotenv import load_dotenv
from langchain_classic.retrievers import BM25Retriever
from langchain_classic.schema import Document
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings

load_dotenv()

True

# Get existing vectorstore

In [2]:
api_key = os.getenv("API_KEY")
collection_name = "langchain_docs_index"
embedding = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001", api_key=api_key)

vectorstore = Chroma(embedding_function=embedding, collection_name=collection_name, persist_directory="./data/vectors/chroma_db")

In [3]:
# Pull all documents from the vectorstore
all_docs = vectorstore.get(include=['documents', 'metadatas'])

# Reconstruct Document objects for BM25
docs_from_chroma = [
    Document(page_content=text, metadata=meta) 
    for text, meta in zip(all_docs['documents'], all_docs['metadatas'])
]

# Initialize retriever

In [4]:
keywordk_retriever = BM25Retriever.from_documents(docs_from_chroma)
keywordk_retriever.k = 2

In [5]:
def keyword_document_search(query: str, k: int) -> list[Document]:
    keywordk_retriever.k = k
    return keywordk_retriever.invoke(query)

In [ ]:
relevant_keyword_documents = keyword_document_search(
    query="What is BPE?",
    k=3,
)

In [ ]:
print("Keyword search results: \n")
for i, document in enumerate(relevant_keyword_documents):
    print(f"Page of the document retrieved: {vars(document)['metadata']['page']}")
    print("---")

Keyword search results:
Page of the document retrieved: 15
---
Page of the document retrieved: 0
---
Page of the document retrieved: 9
---


# Use Cohere Rerank

In [17]:
api_key = os.getenv("COHERE_API_KEY")
co = cohere.Client(api_key)

In [28]:
reranked_hits = co.rerank(
    query="What is BPE?",
    documents=[doc.page_content for doc in relevant_keyword_documents],
    top_n=10,
    model="rerank-multilingual-v3.0",
)

In [36]:
print("Reranked results: \n")
for hit in reranked_hits.results:
    print(f"Page of the document retrieved: {relevant_keyword_documents[hit.index].metadata['page']}")
    print("---")

Reranked results: 

Page of the document retrieved: 9
---
Page of the document retrieved: 15
---
Page of the document retrieved: 0
---
